PC is program counter and it stores VA, not PA!

say this is trampoline code:

```
# PA = 0x9000
csrw satp, t1

# PA = 0x9004
addi a0, a0, 1
```

we switch page table from old to new(t1), so does the new table also have:

```
# PA = 0x9004
addi a0, a0, 1`
```
at that address? 

Yes! that's the point of trampoline. It resides in the same VA of user and kernel space. Those VA maps to the same physical address. So in trampoline, switch page table does not affect the execution of code.

---

Key issue: updating page table between user/kernel space could disrupte PC and code execution.

trampoline is 1 solution.

Don't switch page table, letting page table contains all user and kernal mapping, is another solution. But it is open to **CPU meltdown attack**.

---

# Meltdown Attack — Simplified Notes

## 1. The Original Design

A process's page table contains both user and kernel memory mappings.

- User pages: `PTE_U = 1`
- Kernel pages: `PTE_U = 0`

User mode cannot read kernel memory because the CPU checks permissions.

**Problem:** Some vulnerable CPUs may temporarily use kernel data before enforcing the permission check.

## 2. The CPU Optimization

Modern CPUs execute instructions speculatively and **out of order** to improve performance.

On a vulnerable CPU:

1. A user program tries to read a kernel address.
2. The CPU **temporarily** obtains the secret value.
3. Dependent instructions may execute using that value.
4. The CPU **detects the permission violation and discards** the results.

**But discarding instruction results does not necessarily undo changes to the CPU cache.**

## 3. The Attack: Turn a Secret into a Cache Signal

Suppose the kernel contains a secret byte:

```c
uint8_t secret = 42;
```

The attacker prepares:

```c
char probe[256 * 4096];
```

Why?

- **256**: One byte has 256 possible values (0–255).
- **4096**: Separates probe locations so they occupy different cache lines.

### Step 1: Flush

Remove the 256 probe locations from the CPU cache.

On x86, attackers can use `clflush`.

### Step 2: Speculative Access

Conceptually, the vulnerable CPU transiently executes:

```c
uint8_t secret = *(uint8_t *)kernel_address;
char x = probe[secret * 4096];
```

If `secret = 42`, the CPU accesses:

```c
probe[42 * 4096];
```

This may bring that location into the CPU cache.

The CPU eventually rejects the unauthorized kernel read, but the cache effect can remain.

### Step 3: Measure

The attacker measures the time needed to read each probe location.

```text
probe[0]   → slow
probe[1]   → slow
...
probe[42]  → FAST!
...
probe[255] → slow
```

Therefore:

```text
secret = 42
```

The attacker never successfully reads the secret directly. Instead, the secret determines which cache line becomes fast to access.

This technique is called **Flush+Reload**.

## 4. The Defense: KPTI

The attack depends on kernel memory being mapped in the user process's page table.

**Kernel Page Table Isolation (KPTI)** removes most kernel mappings from the page table used in user mode.

```text
Before KPTI:
User Page Table = User Mappings + Kernel Mappings

After KPTI:
User Page Table   = User Mappings + Minimal Kernel Entry Mappings
Kernel Page Table = Kernel Mappings + Required User Mappings
```

The OS must now switch page tables when entering or leaving the kernel.

A **trampoline** provides executable code that remains accessible across the switch.